# HtmlParser Demo

Shows flat and tree parsing of an HTML file, including heading hierarchy, table extraction, and the bridge to a flat chunker.

| Aspect | Notes |
|--------|-------|
| **Pros** | Uses built-in `html.parser` — no lxml required; deterministic heading detection via `<h1>`–`<h6>` tags; handles tables, images, paragraphs, and list items; same flat/tree API as all other parsers |
| **Cons** | Images store `src` attribute only (base64 encoding is a future enhancement); malformed HTML may produce unexpected block ordering |
| **vs. MarkdownParser** | HTML heading detection is tag-based and always reliable; Markdown uses regex which requires well-formed ATX syntax |
| **vs. PdfParser / DocxParser** | HTML structure is explicit in the markup, so no heuristics or style inspection are required; however images are not embedded (src string only) |

## Imports

In [ ]:
# Standard Library
import pathlib

# Third Party Library

# Private Library
from cleave.parsers.factory import ParserFactory
from cleave.schemas import ContentType, Document

## Fixture

In [ ]:
FIXTURES = pathlib.Path.cwd().parent.parent.parent / "tests" / "fixtures"
HTML_PATH = str(FIXTURES / "sample.html")

if not (FIXTURES / "sample.html").exists():
    from tests.fixtures.html import create_sample_html
    create_sample_html(FIXTURES / "sample.html")

print("HTML:", HTML_PATH)

| Mode | `Document` field | Use when |
|------|-----------------|----------|
| `flat` | `.pages` — one virtual `DocumentPage` | Simple text retrieval, fixed or sentence chunking |
| `tree` | `.root` — recursive `TreeNode` hierarchy | Semantic / heading-scoped chunking, multimodal pipelines |

## Flat mode

In [ ]:
flat_doc = ParserFactory.create(HTML_PATH, mode="flat").parse()
assert isinstance(flat_doc, Document)
print(f"Source type : {flat_doc.source.type}")
print(f"Pages       : {len(flat_doc.pages)}")
print(f"Blocks      : {len(flat_doc.pages[0].blocks)}")

In [ ]:
for block in flat_doc.pages[0].blocks:
    preview = block.content[:60].replace("\n", "\\n")
    print(f"  [{block.position}] type={block.type.value:<6}  {preview!r}")

In [ ]:
tables = flat_doc.all_tables
print(f"Tables found: {len(tables)}")
print(tables[0].content)

In [ ]:
images = flat_doc.all_images
print(f"Images found: {len(images)}")
# src string stored as content (base64 encoding: TODO)
print(f"Image src   : {images[0].content!r}")

## Tree mode

In [ ]:
tree_doc = ParserFactory.create(HTML_PATH, mode="tree").parse()
assert tree_doc.root is not None
print(f"Root children : {len(tree_doc.root.children)}")

In [ ]:
def print_tree(node, indent=0):
    role = node.metadata.get("role", "")
    level = node.metadata.get("level", "")
    tag = f"{role}" + (f"[{level}]" if level else "")
    preview = node.content[:50].replace("\n", "\\n")
    print(" " * indent + f"{tag:<18}  {preview!r}")
    for child in node.children:
        print_tree(child, indent + 2)

print_tree(tree_doc.root)

In [ ]:
def find_headings(node):
    out = []
    if node.metadata.get("role") == "heading":
        out.append((node.metadata["level"], node.content))
    for child in node.children:
        out.extend(find_headings(child))
    return out

headings = find_headings(tree_doc.root)
print("Headings found:")
for lvl, text in headings:
    print(f"  H{lvl}: {text}")

In [ ]:
table_nodes = tree_doc.root.get_nodes_by_type(ContentType.table)
print(f"Table nodes : {len(table_nodes)}")
print(table_nodes[0].content)

## Bridge - tree to Markdown string

In [ ]:
md_out = tree_doc.to_markdown()
print(md_out)

## full_text comparison

In [ ]:
flat_text = flat_doc.full_text
tree_text = tree_doc.full_text
print(f"Flat full_text length : {len(flat_text)}")
print(f"Tree full_text length : {len(tree_text)}")
print(f"\nFlat preview : {flat_text[:120]!r}")